In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Models
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from xgboost import XGBRegressor

# Setup random seed
np.random.seed(42)

#Import data and drop
# Displaying the dataset
data = pd.read_csv("../dataset/melb_data.csv")

# Renaming the columns
data.rename(columns={
    "Type": "Property_Type",
    "Method": "Sale_Method"}, inplace=True)

# Renaming the data in columns
data["Property_Type"] = data["Property_Type"].replace({"h":"House",
                                                       "u":"Unit",
                                                       "t":"Townhouse"})

data["Sale_Method"] = data["Sale_Method"].replace({"S":"Sold",
                                                   "SP":"Sold_Prior",
                                                   "PI":"Passed_In",
                                                   "VB":"Vendor_Bid",
                                                   "SA":"Sold_After"})

# Filling CouncilArea missing values and freq encoding (Suburbs & CouncilArea)
data["CouncilArea"] = data["CouncilArea"].fillna("Unknown")
council_freq = data["CouncilArea"].value_counts(normalize=True)
data["CouncilArea_freq"] = data["CouncilArea"].map(council_freq)

suburb_freq = data["Suburb"].value_counts(normalize=True)
data["Suburb_freq"] = data["Suburb"].map(suburb_freq)

data = data.drop(columns=["Address", "SellerG","Suburb","CouncilArea"])

# NUMERICAL COLUMNS
data.loc[data["YearBuilt"] < 1800, "YearBuilt"] = None

categorical_features = ["Property_Type","Sale_Method","Regionname"]
categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))])
     
# Adding missing tag columns
data["BuildingArea_missing"] = data["BuildingArea"].isnull().astype(int)
data["YearBuilt_missing"] = data["YearBuilt"].isnull().astype(int)

numeric_features = ["Car","BuildingArea","YearBuilt"]
numeric_transformer = Pipeline(steps=[
    ("Imputer",SimpleImputer(strategy="median"))])


# Extraction of date sale year and sale month
data["Date"] = pd.to_datetime(data["Date"], dayfirst=True)

data["Sale_Year"] = data["Date"].dt.year
data["Sale_Month"] = data["Date"].dt.month
data["Property_Age_At_Sale"] = data["Sale_Year"] - data["YearBuilt"]

# Preprocessor
preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_transformer,categorical_features),
    ("num",numeric_transformer,numeric_features)])

# Clipping Few columns 
for col in ["BuildingArea", "Landsize"]:
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    df[col] = df[col].clip(lower, upper)

#Dropping all the unnecessary columns
dropper = DropColumns(coulumns=["Date",
                                "Postcode", "Bedroom2","Property_Type",
                                "Sale_Method","Regionname"]


In [6]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

# Setup random seed
np.random.seed(42)

# LOAD DATA
data = pd.read_csv("../dataset/melb_data.csv")

# RENAME COLUMNS
data.rename(columns={
    "Type": "Property_Type",
    "Method": "Sale_Method"}, inplace=True)

# CLEAN CATEGORICAL VALUES
data["Property_Type"] = data["Property_Type"].replace({
    "h":"House","u":"Unit","t":"Townhouse"
})

data["Sale_Method"] = data["Sale_Method"].replace({
    "S":"Sold","SP":"Sold_Prior","PI":"Passed_In",
    "VB":"Vendor_Bid","SA":"Sold_After"
})

# HANDLE MISSING + FREQ ENCODING
data["CouncilArea"] = data["CouncilArea"].fillna("Unknown")

council_freq = data["CouncilArea"].value_counts(normalize=True)
data["CouncilArea_freq"] = data["CouncilArea"].map(council_freq)

suburb_freq = data["Suburb"].value_counts(normalize=True)
data["Suburb_freq"] = data["Suburb"].map(suburb_freq)

# DROP HIGH CARDINALITY
data = data.drop(columns=["Address", "SellerG", "Suburb", "CouncilArea"])

# FIX YEAR
data.loc[data["YearBuilt"] < 1800, "YearBuilt"] = None

# MISSING FLAGS
data["BuildingArea_missing"] = data["BuildingArea"].isnull().astype(int)
data["YearBuilt_missing"] = data["YearBuilt"].isnull().astype(int)

# DATE FEATURES
data["Date"] = pd.to_datetime(data["Date"], dayfirst=True)
data["Sale_Year"] = data["Date"].dt.year
data["Sale_Month"] = data["Date"].dt.month
data["Property_Age_At_Sale"] = data["Sale_Year"] - data["YearBuilt"]

# CLIPPING (OUTLIERS)
for col in ["BuildingArea", "Landsize"]:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    data[col] = data[col].clip(lower, upper)

# SPLIT TARGET
X = data.drop("Price", axis=1)
y = data["Price"]

# DROP UNUSED
X = X.drop(columns=["Date","Postcode","Bedroom2"])

# FEATURES (IMPORTANT FIX)
categorical_features = ["Property_Type","Sale_Method","Regionname"]

numeric_features = [
    "Rooms",
    "Distance",
    "Bathroom",
    "Car",
    "Landsize",
    "BuildingArea",
    "YearBuilt",
    "Lattitude",
    "Longtitude",
    "BuildingArea_missing",
    "YearBuilt_missing",
    "Sale_Year",
    "Sale_Month",
    "Property_Age_At_Sale",
    "CouncilArea_freq",
    "Suburb_freq"
]

# TRANSFORMERS
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# PREPROCESSOR
preprocessor = ColumnTransformer(transformers=[
    ("cat", categorical_transformer, categorical_features),
    ("num", numeric_transformer, numeric_features)
])

# MODEL PIPELINE
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8
    ))
])

# TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# TRAIN
model.fit(X_train, y_train)

# PREDICT
y_pred = model.predict(X_test)

# EVALUATE
print("R2 Score:", r2_score(y_test, y_pred))

R2 Score: 0.8525880773682968


In [8]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

np.random.seed(42)

# LOAD DATA
data = pd.read_csv("../dataset/melb_data.csv")

# RENAME
data.rename(columns={"Type": "Property_Type","Method": "Sale_Method"}, inplace=True)

# CLEAN CATEGORICAL
data["Property_Type"] = data["Property_Type"].replace({
    "h":"House","u":"Unit","t":"Townhouse"
})

data["Sale_Method"] = data["Sale_Method"].replace({
    "S":"Sold","SP":"Sold_Prior","PI":"Passed_In",
    "VB":"Vendor_Bid","SA":"Sold_After"
})

# FREQ ENCODING
data["CouncilArea"] = data["CouncilArea"].fillna("Unknown")

council_freq = data["CouncilArea"].value_counts(normalize=True)
data["CouncilArea_freq"] = data["CouncilArea"].map(council_freq)

suburb_freq = data["Suburb"].value_counts(normalize=True)
data["Suburb_freq"] = data["Suburb"].map(suburb_freq)

data = data.drop(columns=["Address", "SellerG","Suburb","CouncilArea"])

# YEAR FIX
data.loc[data["YearBuilt"] < 1800, "YearBuilt"] = None

# FLAGS
data["BuildingArea_missing"] = data["BuildingArea"].isnull().astype(int)
data["YearBuilt_missing"] = data["YearBuilt"].isnull().astype(int)

# DATE FEATURES
data["Date"] = pd.to_datetime(data["Date"], dayfirst=True)
data["Sale_Year"] = data["Date"].dt.year
data["Sale_Month"] = data["Date"].dt.month
data["Property_Age_At_Sale"] = data["Sale_Year"] - data["YearBuilt"]

# CLIPPING
for col in ["BuildingArea", "Landsize"]:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    data[col] = data[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

# SPLIT TARGET
X = data.drop("Price", axis=1)
y = data["Price"]

# DROP UNUSED
X = X.drop(columns=["Date","Postcode","Bedroom2"])

# FEATURES
categorical_features = ["Property_Type","Sale_Method","Regionname"]

numeric_features = [
    "Rooms","Distance","Bathroom","Car","Landsize",
    "BuildingArea","YearBuilt","Lattitude","Longtitude",
    "BuildingArea_missing","YearBuilt_missing",
    "Sale_Year","Sale_Month","Property_Age_At_Sale",
    "CouncilArea_freq","Suburb_freq"
]

# TRANSFORMERS
cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# PREPROCESSOR
preprocessor = ColumnTransformer([
    ("cat", cat_transformer, categorical_features),
    ("num", num_transformer, numeric_features)
])

# PIPELINE
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(random_state=42))
])

# TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# GRIDSEARCH PARAMS
param_grid = {
    "regressor__n_estimators": [200, 300],
    "regressor__max_depth": [3, 5],
    "regressor__learning_rate": [0.05, 0.1],
    "regressor__subsample": [0.8],
    "regressor__colsample_bytree": [0.8]
}

# GRIDSEARCH
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

# TRAIN
grid.fit(X_train, y_train)

# BEST MODEL
best_model = grid.best_estimator_

# PREDICT
y_pred = best_model.predict(X_test)

# RESULT
print("Best Params:", grid.best_params_)
print("R2 Score:", r2_score(y_test, y_pred))

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Params: {'regressor__colsample_bytree': 0.8, 'regressor__learning_rate': 0.1, 'regressor__max_depth': 5, 'regressor__n_estimators': 300, 'regressor__subsample': 0.8}
R2 Score: 0.8525880773682968


In [18]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

np.random.seed(42)

# LOAD DATA
data = pd.read_csv("../dataset/melb_data.csv")

# RENAME
data.rename(columns={
    "Type": "Property_Type",
    "Method": "Sale_Method"
}, inplace=True)

# CLEAN CATEGORICAL
data["Property_Type"] = data["Property_Type"].replace({
    "h":"House","u":"Unit","t":"Townhouse"
})

data["Sale_Method"] = data["Sale_Method"].replace({
    "S":"Sold","SP":"Sold_Prior","PI":"Passed_In",
    "VB":"Vendor_Bid","SA":"Sold_After"
})

# FIX YEAR
data.loc[data["YearBuilt"] < 1800, "YearBuilt"] = np.nan

# DATE FEATURES
data["Date"] = pd.to_datetime(data["Date"], dayfirst=True)
data["Sale_Year"] = data["Date"].dt.year
data["Sale_Month"] = data["Date"].dt.month
data["Property_Age_At_Sale"] = data["Sale_Year"] - data["YearBuilt"]

# MISSING FLAGS
data["BuildingArea_missing"] = data["BuildingArea"].isnull().astype(int)
data["YearBuilt_missing"] = data["YearBuilt"].isnull().astype(int)

# FREQ ENCODING
data["CouncilArea"] = data["CouncilArea"].fillna("Unknown")

council_freq = data["CouncilArea"].value_counts(normalize=True)
data["CouncilArea_freq"] = data["CouncilArea"].map(council_freq)

suburb_freq = data["Suburb"].value_counts(normalize=True)
data["Suburb_freq"] = data["Suburb"].map(suburb_freq)

# DROP IRRELEVANT (BEFORE SPLIT ✅)
data = data.drop(columns=[
    "Address","SellerG","Suburb","CouncilArea",
    "Date","Postcode","Bedroom2"
])

# SPLIT TARGET
X = data.drop("Price", axis=1)
y = data["Price"]

# FEATURES
categorical_features = ["Property_Type","Sale_Method","Regionname"]

numeric_features = [
    "Rooms","Distance","Bathroom","Car","Landsize",
    "BuildingArea","YearBuilt","Lattitude","Longtitude",
    "BuildingArea_missing","YearBuilt_missing",
    "Sale_Year","Sale_Month","Property_Age_At_Sale",
    "CouncilArea_freq","Suburb_freq"
]

# TRANSFORMERS
cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# PREPROCESSOR
preprocessor = ColumnTransformer([
    ("cat", cat_transformer, categorical_features),
    ("num", num_transformer, numeric_features)
])

# PIPELINE
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(random_state=42))
])

# SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# GRIDSEARCH
param_grid = {
    "regressor__n_estimators": [200, 300],
    "regressor__max_depth": [3, 5],
    "regressor__learning_rate": [0.05, 0.1],
    "regressor__subsample": [0.8],
    "regressor__colsample_bytree": [0.8]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=2
)

# TRAIN
grid.fit(X_train, y_train)

# BEST MODEL
best_model = grid.best_estimator_

# PREDICT
y_pred = best_model.predict(X_test)

# RESULT
print("Best Params:", grid.best_params_)
print("R2 Score:", r2_score(y_test, y_pred))

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Params: {'regressor__colsample_bytree': 0.8, 'regressor__learning_rate': 0.1, 'regressor__max_depth': 5, 'regressor__n_estimators': 300, 'regressor__subsample': 0.8}
R2 Score: 0.849555090307626


In [19]:
import pickle

with open("xgb_pipeline.pkl", "wb") as f:
    pickle.dump(best_model, f)